In [ ]:
import pickle
import os
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import numpy as np
import math
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
from time import time
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import torch.optim as optim
from torch.nn import TransformerEncoder
from torch.nn import TransformerDecoder
import matplotlib.pyplot as plt
import dateutil
from datetime import timedelta
from tqdm.notebook import trange
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
import seaborn as sns
# from src.pot import *
#%%
def dataframe_from_csv(target):
    return pd.read_csv(target, sep=';').rename(columns=lambda x: x.strip())

def dataframe_from_csvs(targets):
    return pd.concat([dataframe_from_csv(x) for x in targets])

TEST_DATASET = sorted([x for x in Path("data").glob("test*.csv")])
TRAIN_DATASET = sorted([x for x in Path("data").glob("train*.csv")])
TEST_DF_RAW = dataframe_from_csvs(TEST_DATASET)
TRAIN_DF_RAW = dataframe_from_csvs(TRAIN_DATASET)

# TRAIN_DF_RAW = dataframe_from_csvs(TRAIN_DATASET)
# TRAIN_DF_RAW.reset_index(drop=True,inplace=True)
print(TRAIN_DF_RAW.shape)
TRAIN_DF_RAW.head()
#%%
TIMESTAMP_FIELD = "time"
# ATTACK_FIELD = "Attack"
NOT_VALID_FIELD = ["time", "attack", "attack_P1","attack_P2","attack_P3"]
# ATTACK_DF = TRAIN_DF_RAW["Attack"]
VALID_COLUMNS_IN_TRAIN_DATASET = TRAIN_DF_RAW.columns.drop(NOT_VALID_FIELD)
#%%
def normalize(df, TAG_MIN, TAG_MAX):
    ndf = df.copy()
    for c in df.columns:
        if TAG_MIN[c] == TAG_MAX[c]:
            ndf[c] = df[c] - TAG_MIN[c]
        else:
            ndf[c] = (df[c] - TAG_MIN[c]) / (TAG_MAX[c] - TAG_MIN[c])
    return ndf
#%%
numeric_cols = TRAIN_DF_RAW[VALID_COLUMNS_IN_TRAIN_DATASET] \
                   .select_dtypes(include='number') \
                   .columns

TAG_MIN_TRAIN = TRAIN_DF_RAW[numeric_cols].min()
TAG_MAX_TRAIN = TRAIN_DF_RAW[numeric_cols].max()
TAG_MIN_TEST = TEST_DF_RAW[numeric_cols].min()
TAG_MAX_TEST = TEST_DF_RAW[numeric_cols].max()
TAG_MIN = np.minimum(TAG_MIN_TRAIN, TAG_MIN_TEST)
TAG_MAX = np.maximum(TAG_MAX_TRAIN, TAG_MAX_TEST)

TRAIN_DF = normalize(TRAIN_DF_RAW[numeric_cols], TAG_MIN_TRAIN, TAG_MAX_TRAIN)
TEST_DF = normalize(TEST_DF_RAW[numeric_cols], TAG_MIN_TRAIN, TAG_MAX_TRAIN)
#%%
def boundary_check(df):
    x = np.array(df, dtype=np.float32)
    return np.any(x > 1.0), np.any(x < 0), np.any(np.isnan(x))

# Boundary Check
print(boundary_check(TRAIN_DF))
print(boundary_check(TEST_DF))
#%%
def convert_to_windows(data):
   windows = []; w_size = 10
   for i, g in enumerate(data): 
      if i >= w_size: w = data[i-w_size:i]
      else: w = torch.cat([data[0].repeat(w_size-i, 1), data[0:i]])
      windows.append(w)
   return torch.stack(windows)